In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# Find the repo root and add it to sys.path FIRST
repo = Path.cwd()
while not (repo / "functions").exists() and repo.parent != repo:
    repo = repo.parent
sys.path[:0] = [str(repo), str(Path.cwd().parent.parent)]

import tempfile, py7zr
import geopandas as gpd
import glob, json
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from rasterio.warp import reproject, calculate_default_transform

from constants import *
from functions.plot_hillshade import plot_hillshade
from utils import *
from fp_plotting import *
from ecosystems.cs.cs_plotting import *
from ecosystems.cs.cs_data_process import *


FP_PATH = Path("/Users/kylabazlen/Documents/Climate_Roadmap/Forest/Data/composite_priority_2020COFAP_unpacked/commondata/raster_data/comp_121_int3")


### PLOT Forest Priority LAYER ###

In [ ]:


# Reproject priority raster to match your hillshade/counties CRS
pri_arr, pri_extent = reproject_raster(FP_PATH, TARGET_CRS)

# Mask background — 0 is nodata for this raster, also catch any float sentinels/NaN
pri_arr = np.ma.masked_where(
    (pri_arr == 0) | (pri_arr == -3.4028234663852886e+38) | np.isnan(pri_arr),
    pri_arr,
)

step = 2
pri_small = pri_arr[::step, ::step]

cmap = plt.get_cmap("Spectral_r")
norm = Normalize(vmin=float(pri_small.min()), vmax=float(pri_small.max()))

# Hillshade base
fig, ax = plot_hillshade()

# Priority layer on top
im = ax.imshow(
    pri_small, extent=pri_extent, cmap=cmap, norm=norm,
    interpolation="nearest", alpha=0.75, zorder=2,
)
cbar = fig.colorbar(im, ax=ax, shrink=0.6, label="Composite Priority (low → high)")
cbar.set_ticks([])

# Counties on top
plt_counties(COUNTIES_PATH, county_edgecolor="black", county_linewidth=1.0, ax=ax)

plt.tight_layout()
plt.show()

### PLOT Bivariate ###

In [2]:
# ============================================================
# Pipeline
# ============================================================

LAYERS_TO_PLOT = ["5 day max precip", "drought__D3", "Days Exceeding 95th Percentile Maximum Temperature"] #["drought__D3"] #["5 day max precip"]
#, "Days Exceeding 95th Percentile Maximum Temperature"
TARGET_CRS = "EPSG:3857"
N_CLASSES = 3

OUT_ROOT = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/fp_bivariate")

# CS = fine reference grid — build once
cs_transform, cs_width, cs_height = build_reference_grid(str(FP_PATH), dst_crs=TARGET_CRS)
cs_aligned = reproject_to_grid(
    str(FP_PATH), cs_transform, cs_width, cs_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)
extent = grid_to_extent(cs_transform, cs_width, cs_height)

selected = LAYERS_TO_PLOT or list(layers.keys())

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue

    for fp in files:
        # Upsample climate to the CS grid
        clim_aligned = reproject_to_grid(
            fp, cs_transform, cs_width, cs_height,
            dst_crs=TARGET_CRS, resampling=Resampling.bilinear,
        )

        # Classify
        (cs_valid, clim_valid), mask_2d = mask_valid_pixels(cs_aligned, clim_aligned)
        x_breaks, y_breaks, codes = bivariate_classify(
            cs_valid, clim_valid, n_classes=N_CLASSES, method="quantile"
        )
        codes_2d = unflatten_to_raster(codes, mask_2d, fill=-1)

        # Save everything
        run_dir = OUT_ROOT / f"{layer_name}__{Path(fp).stem}"
        save_bivariate_outputs(
            run_dir,
            bivar_map=codes_2d,
            extent=extent,
            color_dict=layers[layer_name].get("bivar"),
            x_label="Composite Forest Priority →",
            y_label=f"{cfg.get('label', layer_name)} →",
            n_classes=N_CLASSES,
            x_breaks=x_breaks,
            y_breaks=y_breaks,
            meta_extra={
                "layer_name": layer_name,
                "climate_file": str(fp),
                "cs_file": str(FP_PATH),
                "x_variable": "Composite Forest Priority",
                "y_variable": cfg.get("label", layer_name),
                "y_units": cfg.get("cbar_label", ""),
                "classification_method": "quantile",
            },
        )
        print(f"✓ saved {run_dir.name}/")

✓ saved 5 day max precip__Rx5day_GWL11C_minus_REF_relative_change_v2/
✓ saved 5 day max precip__Rx5day_GWL15C_minus_REF_relative_change_v2/
✓ saved 5 day max precip__Rx5day_GWL20C_minus_REF_relative_change_v2/
✓ saved 5 day max precip__Rx5day_GWL25C_minus_REF_relative_change_v2/
✓ saved drought__D3__pct_change_Aridity_GWL11C_D3_timescale_12months/
✓ saved drought__D3__pct_change_Aridity_GWL15C_D3_timescale_12months/
✓ saved drought__D3__pct_change_Aridity_GWL20C_D3_timescale_12months/
✓ saved drought__D3__pct_change_Aridity_GWL25C_D3_timescale_12months/
✓ saved Days Exceeding 95th Percentile Maximum Temperature__TX95p_GWL11C_minus_REF_absoule_change_v2/
✓ saved Days Exceeding 95th Percentile Maximum Temperature__TX95p_GWL15C_minus_REF_absoule_change_v2/
✓ saved Days Exceeding 95th Percentile Maximum Temperature__TX95p_GWL20C_minus_REF_absoule_change_v2/
✓ saved Days Exceeding 95th Percentile Maximum Temperature__TX95p_GWL25C_minus_REF_absoule_change_v2/
